# Analise Exploratoria de Dados (EDA)
## Pipeline de Risco de Credito
**Autora:** Nayane Araujo | [github.com/Nayanearaujo](https://github.com/Nayanearaujo)

---
Este notebook explora os dados de credito para entender o perfil dos tomadores,
identificar padroes de inadimplencia e planejar a limpeza dos dados.

**Dataset:** 32.581 registros de emprestimos pessoais com 12 variaveis.

## 1. Importacoes e Configuracao

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

DOCS = Path('../docs')
DOCS.mkdir(parents=True, exist_ok=True)

print('Bibliotecas carregadas!')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'seaborn : {sns.__version__}')

## 2. Carregamento dos Dados (Bronze)

In [ ]:
BRONZE = Path('../data/bronze/credit_risk_raw.csv')
df_raw = pd.read_csv(BRONZE)
print(f'Dataset carregado: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas')
print(f'Tamanho em memoria: {df_raw.memory_usage(deep=True).sum()/1024:.0f} KB')

In [ ]:
print('Primeiras linhas do dataset:')
display(df_raw.head())

In [ ]:
print('Estatisticas descritivas:')
display(df_raw.describe().round(2))

## 3. Analise de Valores Nulos

In [ ]:
nulls = df_raw.isnull().sum()
null_pct = (nulls / len(df_raw) * 100).round(2)
null_report = pd.DataFrame({'Coluna': nulls.index, 'Qtd Nulos': nulls.values, 'Percentual (%)': null_pct.values})
null_report = null_report[null_report['Qtd Nulos'] > 0].sort_values('Qtd Nulos', ascending=False)

print('Colunas com valores nulos:')
if len(null_report) == 0:
    print('Nenhuma!')
else:
    print(null_report.to_string(index=False))

if len(null_report) > 0:
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.barh(null_report['Coluna'], null_report['Percentual (%)'], color='#e74c3c')
    ax.set_xlabel('% de valores nulos')
    ax.set_title('Valores Nulos por Coluna', fontweight='bold')
    for i, (_, row) in enumerate(null_report.iterrows()):
        ax.text(row['Percentual (%)'] + 0.1, i, f"{row['Percentual (%)']}%", va='center')
    plt.tight_layout()
    plt.savefig(DOCS / 'eda_nulls.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Grafico salvo em docs/eda_nulls.png')

## 4. Distribuicao do Target (loan_status)
- **0 = Adimplente**: pagou em dia
- **1 = Inadimplente**: nao pagou (calote)

In [ ]:
counts = df_raw['loan_status'].value_counts()
pct = df_raw['loan_status'].value_counts(normalize=True) * 100

print(f'Adimplente  (0): {counts[0]:,} ({pct[0]:.1f}%)')
print(f'Inadimplente(1): {counts[1]:,} ({pct[1]:.1f}%)')
print(f'Razao: {counts[0]/counts[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cores = ['#2ecc71', '#e74c3c']
labels = ['Adimplente (0)', 'Inadimplente (1)']

axes[0].pie([counts[0], counts[1]], labels=labels, colors=cores,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Proporcao das Classes', fontweight='bold')

bars = axes[1].bar(labels, [counts[0], counts[1]], color=cores, width=0.5)
axes[1].set_title('Quantidade por Classe', fontweight='bold')
axes[1].set_ylabel('Registros')
for bar, v, p in zip(bars, [counts[0], counts[1]], [pct[0], pct[1]]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{v:,}\n({p:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(DOCS / 'eda_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Dataset desbalanceado -> usaremos SMOTE na modelagem.')

## 5. Variaveis Numericas por Status de Inadimplencia

In [ ]:
num_cols = ['person_age', 'person_income', 'person_emp_length',
            'loan_amnt', 'loan_int_rate', 'loan_percent_income',
            'cb_person_cred_hist_length']

fig, axes = plt.subplots(len(num_cols), 2, figsize=(14, 4 * len(num_cols)))

for i, col in enumerate(num_cols):
    for status, cor, nome in [(0, '#2ecc71', 'Adimplente'), (1, '#e74c3c', 'Inadimplente')]:
        dados = df_raw[df_raw['loan_status'] == status][col].dropna()
        axes[i, 0].hist(dados, bins=30, alpha=0.6, color=cor, label=nome, density=True)
    axes[i, 0].set_title(f'Distribuicao: {col}', fontweight='bold')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].legend()

    d0 = df_raw[df_raw['loan_status'] == 0][col].dropna()
    d1 = df_raw[df_raw['loan_status'] == 1][col].dropna()
    bp = axes[i, 1].boxplot([d0, d1], patch_artist=True, labels=['Adimplente', 'Inadimplente'])
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    axes[i, 1].set_title(f'Boxplot: {col}', fontweight='bold')

plt.suptitle('Variaveis Numericas por Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(DOCS / 'eda_numericas.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafico salvo em docs/eda_numericas.png')

## 6. Variaveis Categoricas — Taxa de Inadimplencia

In [ ]:
cat_cols = ['loan_grade', 'loan_intent', 'person_home_ownership', 'cb_person_default_on_file']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    taxa = df_raw.groupby(col)['loan_status'].agg(['mean','count']).reset_index()
    taxa.columns = ['cat', 'taxa', 'total']
    taxa['pct'] = taxa['taxa'] * 100
    taxa = taxa.sort_values('pct', ascending=False)

    cores_map = plt.cm.RdYlGn_r(taxa['pct'] / (taxa['pct'].max() + 0.001))
    bars = axes[idx].bar(taxa['cat'], taxa['pct'], color=cores_map)
    axes[idx].set_title(f'Inadimplencia por {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Taxa (%)')
    axes[idx].set_ylim(0, taxa['pct'].max() * 1.25)
    for bar, (_, row) in zip(bars, taxa.iterrows()):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                       f'{row["pct"]:.1f}%\nn={row["total"]:,}', ha='center', fontsize=8)

plt.suptitle('Taxa de Inadimplencia por Variavel Categorica', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS / 'eda_categoricas.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafico salvo em docs/eda_categoricas.png')

## 7. Mapa de Correlacao

In [ ]:
df_num = df_raw[num_cols + ['loan_status']].copy()
corr = df_num.corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, ax=ax, linewidths=0.5)
ax.set_title('Mapa de Correlacao', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS / 'eda_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelacao com o target (loan_status):')
corr_target = corr['loan_status'].drop('loan_status').sort_values(ascending=False)
for col, val in corr_target.items():
    print(f'  {col:35s}: {val:+.3f}')

## 8. Resumo e Conclusoes

In [ ]:
print('=' * 60)
print('RESUMO DA ANALISE EXPLORATORIA')
print('=' * 60)
print(f'Total de registros : {len(df_raw):,}')
print(f'Total de variaveis : {len(df_raw.columns)}')
print(f'Taxa de inadimplencia: {df_raw["loan_status"].mean()*100:.1f}%')
print(f'Colunas com nulos  : {(df_raw.isnull().sum()>0).sum()}')
print()

print('PRINCIPAIS INSIGHTS:')
grade_taxa = df_raw.groupby('loan_grade')['loan_status'].mean() * 100
print(f'  Grade A: {grade_taxa["A"]:.1f}% inadimplencia')
print(f'  Grade G: {grade_taxa["G"]:.1f}% inadimplencia')

def_yes = df_raw[df_raw['cb_person_default_on_file']=='Y']['loan_status'].mean()*100
def_no  = df_raw[df_raw['cb_person_default_on_file']=='N']['loan_status'].mean()*100
print(f'  Historico de calote: {def_yes:.1f}% vs sem historico: {def_no:.1f}%')

high_commit = df_raw[df_raw['loan_percent_income']>0.3]['loan_status'].mean()*100
low_commit  = df_raw[df_raw['loan_percent_income']<=0.3]['loan_status'].mean()*100
print(f'  Comprometimento > 30%: {high_commit:.1f}% vs <= 30%: {low_commit:.1f}%')

print()
print('PROXIMOS PASSOS:')
print('  -> Notebook 02: Feature Engineering')
print('  -> Notebook 03: Treinamento dos Modelos ML')
print('=' * 60)

graficos = list(DOCS.glob('eda_*.png'))
print(f'\nGraficos gerados ({len(graficos)}):')
for g in sorted(graficos):
    print(f'  {g.name}')